In [ ]:
# Notebook 2/9 — Retracted OA PDF Downloader (CORE API Top-Up)
#
# Purpose: This notebook downloads additional Open Access (OA) PDFs for retracted articles that are still missing
# after the primary Unpaywall download step.
#
# What this notebook does:
# 1) Loads a DOI list from Google Drive (semicolon-separated CSV with columns: code, doi).
# 2) Skips PDFs already present in the output folder.
# 3) Skips cases already attempted (success or failure) as recorded in the log.
# 4) Uses the CORE API to search by DOI and retrieve candidate full-text/download URLs.
# 5) Downloads the PDF using a safe-write approach (.part then rename) and validates the PDF signature.
# 6) Appends a row to a persistent CSV log after each attempt (resume-safe).
#
# Inputs:
# - /content/drive/MyDrive/Dissertation/retracted_dois.csv
#   Columns (required): code ; doi
#
# Outputs:
# - PDFs saved to: /content/drive/MyDrive/Dissertation/oa_pdfs_all/<code>.pdf
# - Log saved to:  /content/drive/MyDrive/Dissertation/core_download_log.csv
#
# Execution:
# - Run top-to-bottom.
# - If the Colab session restarts, rerun the notebook; the log and existing PDFs prevent duplicate work.
#
# Notes:
# - CORE API key is requested interactively (hidden input).
# - This notebook is designed to be used as a “top-up” stage in the full pipeline.
# - This notebook is part of a larger, multi-stage data acquisition and
# analysis pipeline and is designed to be fully reproducible.

In [ ]:
# 1) Mount Google Drive + Install dependencies

from google.colab import drive
drive.mount("/content/drive")

!pip -q install pandas requests tqdm tenacity

In [ ]:
# 2) Imports

import os, re, time, csv, requests
import pandas as pd
from tqdm import tqdm
from getpass import getpass
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

In [ ]:
# 2) Configuration

DISSERTATION_DIR = "/content/drive/MyDrive/Dissertation"
INPUT_CSV = os.path.join(DISSERTATION_DIR, "retracted_dois.csv")
OUT_DIR = os.path.join(DISSERTATION_DIR, "oa_pdfs_all")
LOG_PATH = os.path.join(DISSERTATION_DIR, "core_download_log.csv")

os.makedirs(OUT_DIR, exist_ok=True)

print("INPUT_CSV:", INPUT_CSV)
print("OUT_DIR:", OUT_DIR)
print("LOG_PATH:", LOG_PATH)

In [ ]:
# 3) Enter CORE API key (secure)

CORE_API_KEY = getpass("Paste CORE API key (hidden): ").strip()
os.environ["CORE_API_KEY"] = CORE_API_KEY

print("CORE_API_KEY loaded for this session.")

In [ ]:
# 4) Helpers (filename safety, PDF validation, HTTP retries)

MIN_VALID_FILESIZE = 10_000
SLEEP_S = 0.25

def normalize_doi(d):
    if d is None:
        return None
    s = str(d).strip()
    if not s:
        return None
    s = s.replace("https://doi.org/", "").replace("http://doi.org/", "")
    return s.strip().lower()

def extract_code_num(code):
    """
    PDFs are named like 47504.pdf.
    If code is '47504' -> '47504'
    If code has extra chars -> extracts first digit group.
    """
    s = str(code).strip()
    m = re.search(r"(\d+)", s)
    return m.group(1) if m else s

def is_good_pdf(path):
    if not os.path.exists(path):
        return False
    if os.path.getsize(path) < MIN_VALID_FILESIZE:
        return False
    try:
        with open(path, "rb") as f:
            return f.read(5) == b"%PDF-"
    except Exception:
        return False

@retry(reraise=True, stop=stop_after_attempt(5),
       wait=wait_exponential(multiplier=1, min=1, max=20),
       retry=retry_if_exception_type(requests.RequestException))
def http_get(url, headers=None, params=None, timeout=60, stream=False):
    return requests.get(
        url,
        headers=headers or {},
        params=params,
        timeout=timeout,
        stream=stream,
        allow_redirects=True,
    )

def download_pdf(url, out_path):
    """
    Downloads url to out_path using .part then atomic rename.
    Validates PDF header.
    """
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/pdf,application/octet-stream;q=0.9,*/*;q=0.8",
    }
    tmp = out_path + ".part"

    r = http_get(url, headers=headers, timeout=90, stream=True)
    r.raise_for_status()

    with open(tmp, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 256):
            if chunk:
                f.write(chunk)

    with open(tmp, "rb") as f:
        if f.read(5) != b"%PDF-":
            os.remove(tmp)
            return False, "Not a PDF"

    os.replace(tmp, out_path)
    return True, "OK"

In [ ]:
# 5) Load input CSV (BOM-safe; force strings)

df = pd.read_csv(
    INPUT_CSV,
    sep=";",
    encoding="utf-8-sig",
    dtype={"code": "string", "doi": "string"}
)
df.columns = [c.strip().replace("\ufeff", "") for c in df.columns]

if "code" not in df.columns or "doi" not in df.columns:
    raise ValueError(f"Expected columns 'code' and 'doi'. Found: {df.columns.tolist()}")

df["code"] = df["code"].astype("string").str.strip()
df["doi"] = df["doi"].apply(normalize_doi)
df["code_num"] = df["code"].apply(extract_code_num)

print("Rows:", len(df))
print("Example codes:", df["code"].head(5).tolist())
print("Example code_num:", df["code_num"].head(5).tolist())
print("Example dois:", df["doi"].head(5).tolist())

In [ ]:
# 6) Folder inventory + compute missing cases

existing_stems = set(
    os.path.splitext(f)[0].strip()
    for f in os.listdir(OUT_DIR)
    if f.lower().endswith(".pdf")
)

df["already_in_folder"] = df["code_num"].isin(existing_stems)

missing = df[~df["already_in_folder"]].copy()

print("PDFs found in folder:", len(existing_stems))
print("Already in folder:", int(df["already_in_folder"].sum()))
print("Missing:", len(missing))

In [ ]:
# 7) Prepare / load log (RESUME-SAFE)

LOG_COLS = ["code","code_num","doi","status","source","pdf_url","filepath","message"]

if not os.path.exists(LOG_PATH):
    pd.DataFrame(columns=LOG_COLS).to_csv(LOG_PATH, index=False)

log = pd.read_csv(LOG_PATH)

# Treat anything already attempted as done for resume purposes
treated_codes_logged = set(
    log.loc[log["status"].isin(["downloaded", "failed"]), "code_num"]
       .astype(str).str.strip().tolist()
)

downloaded_codes_logged = set(
    log.loc[log["status"] == "downloaded", "code_num"]
       .astype(str).str.strip().tolist()
)

print("Already logged attempted (downloaded+failed):", len(treated_codes_logged))
print("Already logged downloaded:", len(downloaded_codes_logged))

In [ ]:
# 8) CORE API resolver + DOI search

CORE_API_KEY = os.getenv("CORE_API_KEY", "").strip()
if not CORE_API_KEY:
    raise ValueError("CORE_API_KEY not found. Run cell 2.")

CORE_API_BASE = "https://api.core.ac.uk/v3"

def core_get_json(endpoint, params=None, timeout=30):
    """
    Tries both common CORE auth styles.
    Returns (json, auth_used, status_code).
    """
    url = endpoint if endpoint.startswith("http") else f"{CORE_API_BASE}{endpoint}"
    headers_bearer = {"Authorization": f"Bearer {CORE_API_KEY}"}
    headers_xkey = {"X-CORE-API-Key": CORE_API_KEY}

    r = requests.get(url, params=params, headers=headers_bearer, timeout=timeout)
    if r.status_code not in (401, 403):
        r.raise_for_status()
        return r.json(), "Authorization: Bearer", r.status_code

    r2 = requests.get(url, params=params, headers=headers_xkey, timeout=timeout)
    r2.raise_for_status()
    return r2.json(), "X-CORE-API-Key", r2.status_code

def looks_like_pdf(u: str) -> bool:
    if not u:
        return False
    u2 = u.lower()
    return (".pdf" in u2) or u2.endswith("/pdf") or u2.endswith("/pdf/")

def extract_urls_from_core_record(rec: dict):
    """
    CORE records vary. Any plausible fulltext/download URLs are extracted.
    """
    urls = []

    # Direct common fields
    for k in ("downloadUrl", "fullTextUrl", "pdfUrl", "url"):
        v = rec.get(k)
        if isinstance(v, str) and v:
            urls.append(v)

    # Nested link collections (common patterns)
    for k in ("links", "fullText", "fullTextLinks", "repositories"):
        v = rec.get(k)
        if isinstance(v, list):
            for it in v:
                if isinstance(it, dict):
                    for kk in ("url", "downloadUrl", "link", "pdfUrl", "fullTextUrl"):
                        vv = it.get(kk)
                        if isinstance(vv, str) and vv:
                            urls.append(vv)
        elif isinstance(v, dict):
            for kk in ("url", "downloadUrl", "pdf", "link", "pdfUrl", "fullTextUrl"):
                vv = v.get(kk)
                if isinstance(vv, str) and vv:
                    urls.append(vv)

    # De-dupe preserve order
    seen = set()
    out = []
    for u in urls:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out

def core_candidate_urls_by_doi(doi: str, limit=5):
    """
    Returns ([(source, url), ...], debug_info)
    """
    doi = normalize_doi(doi)
    if not doi:
        return [], {"reason": "missing_doi"}

    endpoints = [
        ("/search/works", {"q": f"doi:{doi}", "limit": limit}),
        ("/search/works", {"q": doi, "limit": limit}),
    ]

    last_err = None
    for ep, params in endpoints:
        try:
            js, auth_used, _ = core_get_json(ep, params=params, timeout=30)

            results = None
            if isinstance(js, dict):
                if isinstance(js.get("results"), list):
                    results = js["results"]
                elif isinstance(js.get("data"), dict) and isinstance(js["data"].get("results"), list):
                    results = js["data"]["results"]

            if not results:
                continue

            candidates = []
            for rec in results[:limit]:
                if not isinstance(rec, dict):
                    continue
                urls = extract_urls_from_core_record(rec)
                # Prefer URLs that look like PDFs first
                pdf_first = [u for u in urls if looks_like_pdf(u)]
                other = [u for u in urls if u not in pdf_first]
                ordered = pdf_first + other
                for u in ordered:
                    candidates.append(("core", u))

            # De-dupe preserve order
            seen = set()
            out = []
            for src, u in candidates:
                if u and u not in seen:
                    seen.add(u)
                    out.append((src, u))

            return out, {"auth": auth_used, "endpoint": ep, "params": params, "results_n": len(results)}
        except Exception as e:
            last_err = str(e)
            continue

    return [], {"reason": "no_results", "last_error": last_err}

In [ ]:
# 9) Run CORE top-up downloader (RESUME-SAFE)

downloaded_n = 0
skipped_exists = 0
skipped_logged = 0
failed_no_url = 0
failed_dl = 0

# How often to force an fsync (Drive safety)
LOG_FLUSH_EVERY = 1
_append_count = 0

def append_log_row(row_dict):
    global _append_count
    file_exists = os.path.exists(LOG_PATH) and os.path.getsize(LOG_PATH) > 0
    row = {c: row_dict.get(c, "") for c in LOG_COLS}

    with open(LOG_PATH, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=LOG_COLS)
        if not file_exists:
            w.writeheader()
        w.writerow(row)

        _append_count += 1
        if LOG_FLUSH_EVERY and (_append_count % LOG_FLUSH_EVERY == 0):
            try:
                f.flush()
                os.fsync(f.fileno())
            except Exception:
                pass

pbar = tqdm(total=len(missing), desc="CORE missing cases", unit="paper")

for _, row in missing.iterrows():
    code = str(row["code"]).strip()
    code_num = str(row["code_num"]).strip()
    doi = row["doi"]

    out_path = os.path.join(OUT_DIR, f"{code_num}.pdf")

    # 1) Skip if file already exists
    if os.path.exists(out_path):
        skipped_exists += 1
        pbar.update(1)
        continue

    # 2) Skip if already attempted (downloaded OR failed) in previous runs
    if code_num in treated_codes_logged:
        skipped_logged += 1
        pbar.update(1)
        continue

    # 3) Missing DOI
    if not doi:
        append_log_row({
            "code": code, "code_num": code_num, "doi": doi,
            "status": "failed", "source": "core", "pdf_url": "",
            "filepath": out_path, "message": "Missing DOI"
        })
        treated_codes_logged.add(code_num)
        failed_no_url += 1
        pbar.update(1)
        continue

    # 4) CORE candidate URLs
    cand, dbg = core_candidate_urls_by_doi(doi, limit=5)

    if not cand:
        append_log_row({
            "code": code, "code_num": code_num, "doi": doi,
            "status": "failed", "source": "core", "pdf_url": "",
            "filepath": out_path, "message": f"No CORE URLs. Debug: {dbg}"
        })
        treated_codes_logged.add(code_num)
        failed_no_url += 1
        pbar.update(1)
        time.sleep(SLEEP_S)
        continue

    # 5) Try downloads until success
    success = False
    last_msg = ""
    last_url = ""

    for src, url in cand:
        try:
            ok, msg = download_pdf(url, out_path)
            if ok:
                append_log_row({
                    "code": code, "code_num": code_num, "doi": doi,
                    "status": "downloaded", "source": src, "pdf_url": url,
                    "filepath": out_path, "message": msg
                })
                treated_codes_logged.add(code_num)
                downloaded_n += 1
                print(f"Saved: {out_path}")
                success = True
                break
            else:
                last_msg = msg
                last_url = url
        except Exception as e:
            last_msg = str(e)
            last_url = url

        time.sleep(SLEEP_S)

    if not success:
        append_log_row({
            "code": code, "code_num": code_num, "doi": doi,
            "status": "failed", "source": "core", "pdf_url": last_url,
            "filepath": out_path, "message": last_msg
        })
        treated_codes_logged.add(code_num)
        failed_dl += 1

    pbar.set_postfix({
        "downloaded": downloaded_n,
        "skip_exists": skipped_exists,
        "skip_logged": skipped_logged,
        "fail_no_url": failed_no_url,
        "fail_dl": failed_dl
    })
    pbar.update(1)
    time.sleep(SLEEP_S)

pbar.close()

print("\n--- CORE SUMMARY (RESUME-SAFE) ---")
print("Downloaded this run:", downloaded_n)
print("Skipped (exists):", skipped_exists)
print("Skipped (already attempted in log):", skipped_logged)
print("Failed (no CORE URL):", failed_no_url)
print("Failed (download/invalid):", failed_dl)
print("Log:", LOG_PATH)
print("Folder:", OUT_DIR)